# Restrições de string (`StringConstraints`)

Vimos antes algumas restrições de string que podemos definir com um `Field`:

In [1]:
from pydantic import BaseModel, Field, ValidationError

In [2]:
class Model(BaseModel):
    name: str = Field(min_length=2, max_length=5)

Mas há outras restrições — algumas vimos na configuração de modelo — que podemos aplicar campo a campo.

Para isso, adicionamos um objeto `StringConstraints` ao tipo anotado:

In [3]:
from typing import Annotated

from pydantic import StringConstraints

In [4]:
StandardString = Annotated[
    str,
    StringConstraints(to_lower=True, min_length=2, strip_whitespace=True)
]

In [5]:
class Model(BaseModel):
    code: StandardString | None = None

In [6]:
print(f"Model(): {Model()!r}")

Model(): Model(code=None)


In [7]:
print(f'Model(code="4111 1111 1111 1111    "): {Model(code="4111 1111 1111 1111    ")!r}')

Model(code="4111 1111 1111 1111    "): Model(code='4111 1111 1111 1111')


In [8]:
print(f'Model(code="ABC   "): {Model(code="ABC   ")!r}')

Model(code="ABC   "): Model(code='abc')


Repare que o tamanho mínimo é verificado **depois** de remover os espaços: `"   a   "` vira `"a"`, que tem só 1 caractere.

In [9]:
try:
    Model(code="   a   ")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
code
  String should have at least 2 characters [type=string_too_short, input_value='   a   ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


Então: se você só precisa de tamanho mínimo/máximo (ou um padrão regex), basta um `Field`; se precisar de algo um pouco mais avançado,
`StringConstraints` pode ser o caminho (e, se não bastar, validadores customizados).

Documentação [aqui](https://docs.pydantic.dev/latest/api/types/#pydantic.types.StringConstraints).